<a href="https://colab.research.google.com/github/armandoordonez/AI-Engineering/blob/main/tutorial_langchain_langsmith.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial práctico: LangChain + LangSmith

**LangChain** sirve para construir aplicaciones con LLMs (prompts, cadenas, herramientas, agentes). **LangSmith** sirve para observarlas, depurarlas y evaluarlas.

**Contenido**
1. Instalación
2. Claves y configuración de LangSmith
3. Primera llamada al modelo
4. Prompts y cadenas (LCEL)
5. Salida estructurada
6. Herramientas y agentes
7. RAG básico
8. `@traceable`: trazar tu propio código
9. Evaluación con datasets
10. Memoria conversacional

> Requiere: cuenta en [smith.langchain.com](https://smith.langchain.com) y una API key de Anthropic **u** OpenAI. Usa la API de LangChain 1.x.

## 1. Instalación

In [1]:
%pip install -qU langchain langchain-anthropic langchain-openai langsmith langgraph langchain-community rank_bm25 pypdf python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 2. Claves y configuración de LangSmith

Con `LANGSMITH_TRACING=true`, todo lo que ejecutes con LangChain queda registrado en LangSmith automáticamente.
Las claves se piden con `getpass` para que no queden guardadas en el notebook.

In [1]:
import os
import getpass
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

def set_key_from_secrets(name):
    if not os.environ.get(name):
        try:
            from google.colab import userdata
            os.environ[name] = userdata.get(name)
        except Exception:
            val = getpass.getpass(f"Ingresa tu {name}: ")
            os.environ[name] = val.strip()

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "tutorial-langchain"

# Cargar LangSmith
set_key_from_secrets("LANGSMITH_API_KEY")

# Elige tu proveedor: "google", "anthropic" u "openai"
PROVEEDOR = "google"
if PROVEEDOR == "anthropic":
    key_name = "ANTHROPIC_API_KEY"
elif PROVEEDOR == "openai":
    key_name = "OPENAI_API_KEY"
elif PROVEEDOR == "google":
    key_name = "GOOGLE_API_KEY"
else:
    key_name = f"{PROVEEDOR.upper()}_API_KEY"
set_key_from_secrets(key_name)

## 3. Primera llamada al modelo

In [2]:
from langchain.chat_models import init_chat_model

if PROVEEDOR == "anthropic":
    MODELO = "anthropic:claude-sonnet-4-5"
elif PROVEEDOR == "openai":
    MODELO = "openai:gpt-4o-mini"
elif PROVEEDOR == "google":
    MODELO = "google_genai:gemini-3.5-flash-lite"
else:
    MODELO = PROVEEDOR

model = init_chat_model(MODELO, temperature=0)

resp = model.invoke("Explica qué es un LLM en una frase corta mediante una poesia.")
if isinstance(resp.content, list):
    print(resp.content[0]["text"] if isinstance(resp.content[0], dict) and "text" in resp.content[0] else resp.content)
else:
    print(resp.content)
print(resp.usage_metadata)  # tokens usados

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


/Users/luiscadn/Documents/Proyectos/Universidad/IALangGraph-LangChain/AI-Engineering/.venv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Un cerebro de palabras que navega en el azar,
para unir versos perdidos y tus dudas contestar.
{'input_tokens': 16, 'output_tokens': 24, 'total_tokens': 40, 'input_token_details': {'cache_read': 0}}


👉 Abre LangSmith → proyecto **tutorial-langchain**. Ahí verás esta llamada con su input, output, tokens y latencia.

## 4. Prompts y cadenas (LCEL)
El operador `|` encadena componentes: **prompt → modelo → parser**.

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un profesor experto en {tema}. Responde en español, en máximo 3 frases."),
    ("human", "{pregunta}"),
])

chain = prompt | model | StrOutputParser()
print(chain.invoke({"tema": "finanzas", "pregunta": "¿Qué es el interés compuesto?"}))

/Users/luiscadn/Documents/Proyectos/Universidad/IALangGraph-LangChain/AI-Engineering/.venv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


El interés compuesto es aquel que se calcula sobre el capital inicial y también sobre los intereses acumulados de periodos anteriores. Esto genera un efecto de "bola de nieve", donde tu dinero crece de manera exponencial con el tiempo. Es el concepto clave para construir riqueza a largo plazo gracias a la reinversión constante.


In [5]:
# --- NUESTRA PROPIA CADENA PERSONALIZADA ---
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Definimos un prompt con rol y estructura clara
mi_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "Eres un mentor senior de ingeniería de software y arquitectura de IA. "
        "Cuando te pregunten sobre un concepto técnico: "
        "1. Explícalo en 2 líneas usando una analogía de la vida cotidiana. "
        "2. Da un ejemplo práctico de cuándo usarlo. "
        "Responde en español con tono profesional y motivador."
    )),
    ("human", "¿Qué es y para qué sirve: {concepto}?")
])

# 2. Armamos la cadena con pipes
mi_cadena = mi_prompt | model | StrOutputParser()

# 3. La invocamos pasando el valor de la variable {concepto}
respuesta = mi_cadena.invoke({"concepto": "LangGraph"})
print(respuesta)


¡Excelente pregunta! Entender estas herramientas es un paso clave para dominar el desarrollo de IA moderno.

**1. Explicación con analogía:**
LangGraph es como el plano operacional de una cocina de restaurante profesional, donde diferentes cocineros (agentes de IA) se pasan un pedido de una estación a otra, pudiendo retroceder si un plato necesita corregirse o pausar para pedir la aprobación del chef principal. A diferencia de los flujos lineales tradicionales, te permite construir procesos de IA interactivos, con bucles (ciclos) y memoria del estado actual en todo momento.

**2. Ejemplo práctico de cuándo usarlo:**
Úsalo cuando necesites construir un **sistema complejo de automatización con aprobación humana (Human-in-the-loop)**. Por ejemplo, un asistente de soporte financiero que:
1. Analice el reclamo de un cliente.
2. Si falta información, ejecute un bucle para pedírsela al usuario.
3. Consulte la base de datos para validar la transacción.
4. Si el reembolso supera los $500 USD, *

In [6]:
# Varias entradas en paralelo
respuestas = chain.batch([
    {"tema": "física", "pregunta": "¿Qué es la entropía?"},
    {"tema": "historia", "pregunta": "¿Qué fue la Ilustración?"},
])
for r in respuestas:
    print("-", r, "\n")

- La entropía es una magnitud física que mide el grado de desorden o aleatoriedad de las partículas en un sistema termodinámico. Representa la tendencia natural de la energía a dispersarse, pasando de estados ordenados a estados menos organizados donde ya no se puede realizar trabajo útil. Según la segunda ley de la termodinámica, la entropía del universo siempre aumenta en los procesos espontáneos, lo que marca la flecha del tiempo. 

- La Ilustración fue un movimiento intelectual, filosófico y cultural que floreció en Europa durante el siglo XVIII, conocido como el "Siglo de las Luces". Este pensamiento promovió el uso prioritario de la razón, la ciencia y la educación para combatir la ignorancia, el absolutismo político y el dogmatismo religioso. Sus ideales de libertad, igualdad y división de poderes sentaron las bases ideológicas de las revoluciones modernas y de la democracia contemporánea. 



In [7]:
# Streaming token a token
for chunk in chain.stream({"tema": "IA", "pregunta": "¿Qué es RAG?"}):
    print(chunk, end="", flush=True)

RAG, o Generación Aumentada por Recuperación,

 es una técnica de Inteligencia Artificial que combina la búsqueda de información en fuentes externas con la capacidad de un modelo de lenguaje para

 generar texto. Funciona consultando documentos o bases de datos actualizadas antes de responder, garantizando que la información entreg

ada sea precisa y contextualizada. Esto reduce drásticamente las "alucinaciones" del modelo y permite adaptarlo a conocimientos

 específicos sin necesidad de reentrenarlo.

## 5. Salida estructurada
Pide al modelo un objeto Pydantic en lugar de texto libre.

In [8]:
from pydantic import BaseModel, Field

class Resumen(BaseModel):
    titulo: str
    puntos_clave: list[str] = Field(description="3 a 5 ideas principales")
    sentimiento: str = Field(description="positivo, negativo o neutro")

extractor = model.with_structured_output(Resumen)

texto = '''La empresa reportó ventas récord en el tercer trimestre, impulsadas por
el canal digital. Sin embargo, los costos logísticos subieron 12% y la gerencia
anunció un plan de eficiencia para 2027.'''

r = extractor.invoke(f"Resume este texto:\n{texto}")
r

Resumen(titulo='Resultados del tercer trimestre', puntos_clave=['Ventas récord impulsadas principalmente por el canal digital.', 'Aumento del 12% en los costos logísticos durante el periodo.', 'Anuncio de un nuevo plan de eficiencia gerencial enfocado en 2027.'], sentimiento='neutro')

In [9]:
from pydantic import BaseModel, Field

# 1. Definimos la estructura exacta que queremos recibir
class Resumen(BaseModel):
    titulo: str
    puntos_clave: list[str] = Field(description="5 palabras con las que analizarias este texto")
    sentimiento: str = Field(description="positivo")

# 2. Le decimos al modelo: "Tu respuesta DEBE encajar en esta clase"
extractor = model.with_structured_output(Resumen)

# 3. Texto de prueba
texto = '''La Procuraduría abrió una indagación preliminar contra Armando Benedetti y Angie Rodríguez, exministro del Interior y exdirectora del Dapre del gobierno Petro, por el “Pacto de Valledupar”, como se le llamó al presunto intento del anterior gobierno para que alrededor de 11 exjefes paramilitares, entre ellos Salvatore Mancuso, enlodaran al expresidente Álvaro Uribe, el exministro Germán Vargas Lleras (q.e.p.d.) y el hoy presidente Abelardo de la Espriella.'''

# 4. Invocamos
r = extractor.invoke(f"Analiza este texto criticamente, detecta sesgos:\n{texto}")
r


Resumen(titulo='Análisis crítico y detección de sesgos e imprecisiones', puntos_clave=['Desinformación', 'Sensacionalismo', 'Sesgo político', 'Lenguaje cargado', 'Imprecisión histórica'], sentimiento='negativo')

## 6. Herramientas y agentes
El agente decide cuándo llamar cada herramienta.

In [10]:
try:
    from langchain.agents import create_agent
except ImportError:
    from langgraph.prebuilt import create_react_agent
    def create_agent(model, tools, system_prompt=None, checkpointer=None, **kwargs):
        return create_react_agent(model, tools, prompt=system_prompt, checkpointer=checkpointer, **kwargs)

from langchain_core.tools import tool

@tool
def clima(ciudad: str) -> str:
    """Devuelve el clima actual de una ciudad."""
    return f"En {ciudad} hay 24°C y está soleado."  # aquí llamarías una API real

@tool
def multiplicar(a: float, b: float) -> float:
    """Multiplica dos números."""
    return a * b

agent = create_agent(
    model=model,
    tools=[clima, multiplicar],
    system_prompt="Eres un asistente útil. Usa herramientas cuando las necesites. Responde en español.",
)

result = agent.invoke({"messages": [
    {"role": "user", "content": "¿Qué clima hace en Cali? Y ¿cuánto es 23.5 por 4?"}
]})

for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

¿Qué clima hace en Cali? Y ¿cuánto es 23.5 por 4?
================================== Ai Message ==================================

[]
Tool Calls:
  clima (call_124291)
 Call ID: call_124291
  Args:
    ciudad: Cali
  multiplicar (call_124292)
 Call ID: call_124292
  Args:
    b: 4
    a: 23.5
================================= Tool Message =================================
Name: clima

En Cali hay 24°C y está soleado.
================================= Tool Message =================================
Name: multiplicar

94.0
================================== Ai Message ==================================

[{'type': 'text', 'text': 'En Cali actualmente hay 24°C y está soleado.\n\nPor otro lado, 23.5 multiplicado por 4 es igual a **94**.', 'extras': {'signature': 'EuUBCuIBAWkUfRM61kXGB7t5u+MJHq+8by8k8j4Hl/Wnez63UPzXtsw1vbDRIMuyCF9BXWg4Cw6H1Qjj+bRf4VcotDhYq9+GsA7ggabzbjg1LFUIiuwWRt4z1r4KjdtP+O9k/AXzFGpBq3HE8zIhL

In [4]:
try:
    from langchain.agents import create_agent
except ImportError:
    from langgraph.prebuilt import create_react_agent
    def create_agent(model, tools, system_prompt=None, checkpointer=None, **kwargs):
        return create_react_agent(model, tools, prompt=system_prompt, checkpointer=checkpointer, **kwargs)

from langchain_core.tools import tool

@tool
def clima(ciudad: str) -> str:
    """Devuelve el clima actual de una ciudad."""
    return f"En {ciudad} hay 24°C y está soleado."  # aquí llamarías una API real

@tool
def multiplicar(a: float, b: float) -> float:
    """Multiplica dos números."""
    return a * b

agent = create_agent(
    model=model,
    tools=[clima, multiplicar],
    system_prompt="Eres un asistente útil. Usa herramientas cuando las necesites. Responde en español.",
)

result = agent.invoke({"messages": [
    {"role": "user", "content": "¿Qué clima hace en Cali, Colombia? Y ¿Que clima hace en ShangHai, China? ¿Que clima hace en Brujas, Brucelas?"}
]})

for m in result["messages"]:
    m.pretty_print()

/Users/luiscadn/Documents/Proyectos/Universidad/IALangGraph-LangChain/AI-Engineering/.venv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


/Users/luiscadn/Documents/Proyectos/Universidad/IALangGraph-LangChain/AI-Engineering/.venv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


================================ Human Message =================================

¿Qué clima hace en Cali, Colombia? Y ¿Que clima hace en ShangHai, China? ¿Que clima hace en Brujas, Brucelas?
================================== Ai Message ==================================

[]
Tool Calls:
  clima (call_108633)
 Call ID: call_108633
  Args:
    ciudad: Cali
  clima (call_108634)
 Call ID: call_108634
  Args:
    ciudad: ShangHai
  clima (call_108635)
 Call ID: call_108635
  Args:
    ciudad: Brujas
================================= Tool Message =================================
Name: clima

En Cali hay 24°C y está soleado.
================================= Tool Message =================================
Name: clima

En ShangHai hay 24°C y está soleado.
================================= Tool Message =================================
Name: clima

En Brujas hay 24°C y está soleado.
================================== Ai Message ==================================

[{'type': 'text', 'text': 'Aq

👉 En LangSmith verás el árbol del agente: modelo → herramienta(s) → modelo. Esto ayuda mucho a depurar.

## 7. RAG básico
Responder con base en tus propios documentos. Para no depender de otra API key, aquí se usa un buscador **BM25** (por palabras clave).
En producción normalmente usarías embeddings y una base vectorial (Chroma, FAISS, pgvector...).

In [5]:
%pip install -qU langchain-community rank_bm25 pypdf

Note: you may need to restart the kernel to use updated packages.


In [6]:
from langchain_community.retrievers import BM25Retriever
from langchain_core.tools import tool

try:
    from langchain.agents import create_agent
except ImportError:
    from langgraph.prebuilt import create_react_agent
    def create_agent(model, tools, system_prompt=None, checkpointer=None, **kwargs):
        return create_react_agent(model, tools, prompt=system_prompt, checkpointer=checkpointer, **kwargs)

politicas = [
    "Vacaciones: cada empleado tiene 15 días hábiles de vacaciones al año.",
    "Trabajo remoto: se permite trabajo remoto 2 días por semana, previa aprobación del jefe.",
    "Reembolsos: los reembolsos de gastos se solicitan antes del día 25 de cada mes.",
    "Capacitación: la empresa financia hasta el 70% de cursos relacionados con el cargo.",
]
retriever = BM25Retriever.from_texts(politicas, k=2)

@tool
def buscar_politicas(consulta: str) -> str:
    """Busca en las políticas internas de la empresa."""
    return "\n".join(d.page_content for d in retriever.invoke(consulta))

rag_agent = create_agent(
    model,
    tools=[buscar_politicas],
    system_prompt="Responde SOLO con base en las políticas encontradas. Si no está, di que no sabes.",
)

out = rag_agent.invoke({"messages": [{"role": "user", "content": "¿Cuántos días de remoto puedo hacer?"}]})
content = out["messages"][-1].content
if isinstance(content, list):
    print(content[0]["text"] if isinstance(content[0], dict) and "text" in content[0] else content)
else:
    print(content)

/var/folders/n4/y0x8hjy97p34z9m7p7jds36h0000gn/T/ipykernel_53912/4251208614.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever
/Users/luiscadn/Documents/Proyectos/Universidad/IALangGraph-LangChain/AI-Engineering/.venv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


/Users/luiscadn/Documents/Proyectos/Universidad/IALangGraph-LangChain/AI-Engineering/.venv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Se permite hacer trabajo remoto hasta 2 días por semana, previa aprobación de tu jefe.


**Con PDFs reales** (opcional):
```python
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
docs = PyPDFLoader("mi_documento.pdf").load()
chunks = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100).split_documents(docs)
retriever = BM25Retriever.from_documents(chunks, k=4)
```

In [7]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever

# Usa LAMB.pdf si está en el directorio, o mi_documento.pdf
pdf_path = "LAMB.pdf" if os.path.exists("LAMB.pdf") else "mi_documento.pdf"

if os.path.exists(pdf_path):
    docs = PyPDFLoader(pdf_path).load()
    chunks = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100).split_documents(docs)
    retriever_pdf = BM25Retriever.from_documents(chunks, k=4)
    print(f"Cargado con éxito: {len(chunks)} fragmentos desde '{pdf_path}'")
else:
    print(f"Nota: Coloca un archivo PDF (como '{pdf_path}') en esta carpeta para probar este bloque.")

Cargado con éxito: 122 fragmentos desde 'LAMB.pdf'


## 8. `@traceable`: trazar tu propio código
Así puedes rastrear en LangSmith funciones que no son de LangChain.

In [8]:
from langsmith import traceable

# Aseguramos que 'chain' esté disponible en caso de ejecutar esta celda aislada
if 'chain' not in globals():
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser
    _prompt = ChatPromptTemplate.from_messages([
        ("system", "Eres un profesor experto en {tema}. Responde en español, en máximo 3 frases."),
        ("human", "{pregunta}"),
    ])
    chain = _prompt | model | StrOutputParser()

@traceable
def clasificar(texto: str) -> str:
    return "facturación" if "factura" in texto.lower() else "atención general"

@traceable(name="pipeline_soporte", tags=["tutorial"], metadata={"version": "v1"})
def responder_ticket(texto: str) -> str:
    categoria = clasificar(texto)
    return chain.invoke({"tema": categoria, "pregunta": texto})

print(responder_ticket("No me llegó la factura de septiembre, ¿qué hago?"))

/Users/luiscadn/Documents/Proyectos/Universidad/IALangGraph-LangChain/AI-Engineering/.venv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Primero, revisa la bandeja de spam de tu correo electrónico y verifica que tus datos fiscales registrados sean correctos. Si no aparece, comunícate con el departamento de atención al cliente de la empresa para solicitar el reenvío o la generación de la misma. Recuerda que, por normativas fiscales, existe un plazo límite para emitir facturas, así que no dejes pasar mucho tiempo.


## 9. Evaluación con datasets
1. Creas un dataset con entradas y salidas esperadas.
2. Defines la función a evaluar (*target*) y uno o más evaluadores.
3. Corres el experimento y lo comparas en la interfaz de LangSmith.

In [9]:
from langsmith import Client

client = Client()
DATASET = "preguntas-basicas-tutorial"

if not client.has_dataset(dataset_name=DATASET):
    ds = client.create_dataset(DATASET, description="Dataset de prueba del tutorial")
    client.create_examples(dataset_id=ds.id, examples=[
        {"inputs": {"pregunta": "¿Cuál es la capital de Colombia?"}, "outputs": {"respuesta": "Bogotá"}},
        {"inputs": {"pregunta": "¿Cuánto es 2 + 2?"},                "outputs": {"respuesta": "4"}},
        {"inputs": {"pregunta": "¿En qué continente está Egipto?"},  "outputs": {"respuesta": "África"}},
    ])

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

qa = ChatPromptTemplate.from_template("Responde solo con la respuesta, sin explicación: {pregunta}") | model | StrOutputParser()

def target(inputs: dict) -> dict:
    return {"respuesta": qa.invoke(inputs)}

# Evaluador 1: coincidencia simple
def contiene_respuesta(outputs: dict, reference_outputs: dict) -> bool:
    return reference_outputs["respuesta"].lower() in outputs["respuesta"].lower()

# Evaluador 2: brevedad
def es_breve(outputs: dict) -> bool:
    return len(outputs["respuesta"].split()) <= 5

resultados = client.evaluate(
    target,
    data=DATASET,
    evaluators=[contiene_respuesta, es_breve],
    experiment_prefix="v1",
)
resultados.to_pandas()

View the evaluation results for experiment: 'v1-6de8fe9b' at:
https://smith.langchain.com/o/e91f55d2-0f46-41d5-83d1-2e20caade9dd/datasets/de8bf8b2-8a11-462c-9c29-054f82bb9307/compare?selectedSessions=68a0fd1c-28b1-456b-8f11-1f3c6b24a9f7




/Users/luiscadn/Documents/Proyectos/Universidad/IALangGraph-LangChain/AI-Engineering/.venv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


/Users/luiscadn/Documents/Proyectos/Universidad/IALangGraph-LangChain/AI-Engineering/.venv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


/Users/luiscadn/Documents/Proyectos/Universidad/IALangGraph-LangChain/AI-Engineering/.venv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


,inputs.pregunta,outputs.respuesta,error,reference.respuesta,feedback.contiene_respuesta,feedback.es_breve,execution_time,example_id,id
0,¿Cuál es la capital de Colombia?,Bogotá,None,Bogotá,True,True,0.574382,4aecfa2a-aeab-47e2-81eb-b0e58e4bcd58,01a19a89-0c30-7c9e-9944-c18898240446
1,¿Cuánto es 2 + 2?,4,None,4,True,True,0.659886,55ba3e67-c6b2-48a1-92e0-b24a0e30df1c,01a19a89-0e71-7df1-a6ad-99cf6651ab71
2,¿En qué continente está Egipto?,África,None,África,True,True,0.663850,56fd54b5-fe36-42b9-91cd-48abdd17345b,01a19a89-1107-77a5-9c34-db597e511763


### LLM como juez (para respuestas abiertas)

In [11]:
from pydantic import BaseModel

class Veredicto(BaseModel):
    correcto: bool
    razon: str

juez = model.with_structured_output(Veredicto)

def llm_juez(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    v = juez.invoke(
        f"""Pregunta: {inputs["pregunta"]}
Respuesta de referencia: {reference_outputs["respuesta"]}
Respuesta del modelo: {outputs["respuesta"]}
¿La respuesta del modelo es correcta respecto a la referencia?"""
    )
    return {"key": "llm_juez", "score": v.correcto, "comment": v.razon}

client.evaluate(target, data=DATASET, evaluators=[llm_juez], experiment_prefix="v1-juez")

View the evaluation results for experiment: 'v1-juez-f748e46d' at:
https://smith.langchain.com/o/e91f55d2-0f46-41d5-83d1-2e20caade9dd/datasets/de8bf8b2-8a11-462c-9c29-054f82bb9307/compare?selectedSessions=9b25f886-e3c0-48a3-84d9-5f7556bde4d2




/Users/luiscadn/Documents/Proyectos/Universidad/IALangGraph-LangChain/AI-Engineering/.venv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


/Users/luiscadn/Documents/Proyectos/Universidad/IALangGraph-LangChain/AI-Engineering/.venv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


/Users/luiscadn/Documents/Proyectos/Universidad/IALangGraph-LangChain/AI-Engineering/.venv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


/Users/luiscadn/Documents/Proyectos/Universidad/IALangGraph-LangChain/AI-Engineering/.venv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


/Users/luiscadn/Documents/Proyectos/Universidad/IALangGraph-LangChain/AI-Engineering/.venv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


/Users/luiscadn/Documents/Proyectos/Universidad/IALangGraph-LangChain/AI-Engineering/.venv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


,inputs.pregunta,outputs.respuesta,error,reference.respuesta,feedback.llm_juez,execution_time,example_id,id
0,¿Cuál es la capital de Colombia?,Bogotá,None,Bogotá,True,0.741434,4aecfa2a-aeab-47e2-81eb-b0e58e4bcd58,01a19a89-17da-797b-86be-5ea59ebdc4f0
1,¿Cuánto es 2 + 2?,4,None,4,True,0.620084,55ba3e67-c6b2-48a1-92e0-b24a0e30df1c,01a19a89-2027-782a-bcec-b07ac5583e9c
2,¿En qué continente está Egipto?,África,None,África,True,0.682760,56fd54b5-fe36-42b9-91cd-48abdd17345b,01a19a89-25bd-7360-91e5-c4da292bdfe5


👉 En LangSmith → **Datasets & Experiments** puedes comparar los experimentos lado a lado cuando cambies el prompt o el modelo.

## 10. Memoria conversacional
Con un *checkpointer*, el agente recuerda la conversación de cada `thread_id`.

In [12]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool

try:
    from langchain.agents import create_agent
except ImportError:
    from langgraph.prebuilt import create_react_agent
    def create_agent(model, tools, system_prompt=None, checkpointer=None, **kwargs):
        return create_react_agent(model, tools, prompt=system_prompt, checkpointer=checkpointer, **kwargs)

if 'clima' not in globals():
    @tool
    def clima(ciudad: str) -> str:
        """Devuelve el clima actual de una ciudad."""
        return f"En {ciudad} hay 24°C y está soleado."

agente_memoria = create_agent(model, tools=[clima], checkpointer=InMemorySaver(),
                              system_prompt="Eres un asistente amable. Responde en español.")
config = {"configurable": {"thread_id": "jose-1"}}

agente_memoria.invoke({"messages": [{"role": "user", "content": "Hola, me llamo José y vivo en Cali."}]}, config)
r = agente_memoria.invoke({"messages": [{"role": "user", "content": "¿Cómo me llamo y qué clima hace donde vivo?"}]}, config)
content = r["messages"][-1].content
if isinstance(content, list):
    print(content[0]["text"] if isinstance(content[0], dict) and "text" in content[0] else content)
else:
    print(content)

/Users/luiscadn/Documents/Proyectos/Universidad/IALangGraph-LangChain/AI-Engineering/.venv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


/Users/luiscadn/Documents/Proyectos/Universidad/IALangGraph-LangChain/AI-Engineering/.venv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


/Users/luiscadn/Documents/Proyectos/Universidad/IALangGraph-LangChain/AI-Engineering/.venv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Te llamas José y vives en Cali. En este momento en Cali hay una temperatura de 24°C y está soleado.


## Próximos pasos
- **Prompt Hub**: versiona prompts con `client.push_prompt("mi-prompt", object=prompt)` y recupéralos con `client.pull_prompt("mi-prompt")`.
- **LangGraph**: para flujos con varios pasos, aprobación humana o varios agentes.
- **Producción**: monitoreo, alertas y *online evaluators* en LangSmith sobre el tráfico real.
- Documentación: [docs.langchain.com](https://docs.langchain.com)

In [14]:
# --- PROMPT HUB EN LANGSMITH ---
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langsmith import Client

if 'client' not in globals():
    client = Client()

# 1. Definir y publicar prompt en LangSmith Hub
prompt_hub = ChatPromptTemplate.from_template("Eres un tutor de IA. Explica de forma sencilla en 2 frases: {tema}")
try:
    url_prompt = client.push_prompt("tutor-ia-reto1", object=prompt_hub)
    print(f"Prompt publicado con éxito en LangSmith Hub: {url_prompt}")
except Exception:
    print("Prompt ya existente/versionado en LangSmith Hub: tutor-ia-reto1")

# 2. Recuperar el prompt versionado desde el Hub y utilizarlo
prompt_recuperado = client.pull_prompt("tutor-ia-reto1")
cadena_hub = prompt_recuperado | model | StrOutputParser()
respuesta_hub = cadena_hub.invoke({"tema": "Machine Learning"})
print("Respuesta usando el prompt recuperado del Hub:")
print(respuesta_hub)

Prompt ya existente/versionado en LangSmith Hub: tutor-ia-reto1
Respuesta usando el prompt recuperado del Hub:
El Machine Learning es una rama de la inteligencia artificial que enseña a las computadoras a aprender de los datos por sí mismas, sin necesidad de ser programadas paso a paso. Funciona analizando patrones para tomar decisiones o hacer predicciones cada vez más precisas.
